# 03 — Gold Layer: Load Star Schema to Fabric Warehouse

**Purpose:** Read Silver Delta Tables, generate surrogate keys, populate star schema in Fabric Warehouse.  
**Output:** `dim_customer`, `dim_product`, `dim_date`, `dim_region`, `fact_sales` in the Fabric Warehouse.  
**Schedule:** After notebook 02 completes.


In [ ]:
# ─────────────────────────────────────────────
# 0. Configuration
# ─────────────────────────────────────────────
import pyodbc
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, dayofmonth, dayofweek, weekofyear,
    quarter, date_format, lit, current_timestamp,
    monotonically_increasing_id, row_number, dense_rank
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

LAKEHOUSE_NAME  = "sales_lakehouse"
WAREHOUSE_CONN  = "<your-fabric-warehouse-connection-string>"  # Replace with your warehouse connection
BASE            = f"abfss://{LAKEHOUSE_NAME}@onelake.dfs.fabric.microsoft.com"

print("Notebook ready")

In [ ]:
# ─────────────────────────────────────────────
# 1. Read Silver Tables
# ─────────────────────────────────────────────
df_sales     = spark.read.format("delta").load(f"{BASE}/Tables/silver/sales")
df_customers = spark.read.format("delta").load(f"{BASE}/Tables/silver/customers")
df_products  = spark.read.format("delta").load(f"{BASE}/Tables/silver/products")

print(f"Sales: {df_sales.count():,}")
print(f"Customers: {df_customers.count():,}")
print(f"Products: {df_products.count():,}")

In [ ]:
# ─────────────────────────────────────────────
# 2. Build dim_customer
# ─────────────────────────────────────────────
w_cust = Window.orderBy("customer_id")

dim_customer = (
    df_customers
    .withColumn("customer_key", row_number().over(w_cust))
    .withColumn("is_current", lit(True))
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to",   lit(None).cast("timestamp"))
    .select(
        "customer_key", "customer_id", "customer_name", "segment",
        "country", "state", "city", "postal_code", "email",
        "is_current", "valid_from", "valid_to"
    )
)

dim_customer.show(3, truncate=False)
print(f"dim_customer: {dim_customer.count():,} rows")

In [ ]:
# ─────────────────────────────────────────────
# 3. Build dim_product
# ─────────────────────────────────────────────
w_prod = Window.orderBy("product_id")

dim_product = (
    df_products
    .withColumn("product_key", row_number().over(w_prod))
    .select(
        "product_key", "product_id", "product_name",
        "category", "sub_category", "brand",
        "standard_cost", "list_price"
    )
)

dim_product.show(3, truncate=False)

In [ ]:
# ─────────────────────────────────────────────
# 4. Build dim_date (full calendar table)
# ─────────────────────────────────────────────
from pyspark.sql.functions import sequence, explode, expr

dim_date = (
    spark.sql("SELECT sequence(to_date('2018-01-01'), to_date('2030-12-31'), interval 1 day) AS date_seq")
    .select(explode(col("date_seq")).alias("date"))
    .withColumn("date_key",     date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year",         year(col("date")))
    .withColumn("quarter",      quarter(col("date")))
    .withColumn("month",        month(col("date")))
    .withColumn("month_name",   date_format(col("date"), "MMMM"))
    .withColumn("week",         weekofyear(col("date")))
    .withColumn("day",          dayofmonth(col("date")))
    .withColumn("day_of_week",  dayofweek(col("date")))
    .withColumn("day_name",     date_format(col("date"), "EEEE"))
    .withColumn("is_weekend",   (dayofweek(col("date")).isin([1, 7])))
    .withColumn("yyyymm",       date_format(col("date"), "yyyyMM").cast("int"))
    .withColumn("quarter_label",expr("concat('Q', quarter(date))"))
)

print(f"dim_date: {dim_date.count():,} rows")

In [ ]:
# ─────────────────────────────────────────────
# 5. Build dim_region
# ─────────────────────────────────────────────
dim_region = (
    df_sales.select("region").distinct()
    .withColumn("region_key", row_number().over(Window.orderBy("region")))
    .select("region_key", "region")
)

dim_region.show()

In [ ]:
# ─────────────────────────────────────────────
# 6. Build fact_sales
# ─────────────────────────────────────────────
fact_sales = (
    df_sales
    # Join to get customer_key
    .join(dim_customer.select("customer_id", "customer_key"), on="customer_id", how="left")
    # Join to get product_key
    .join(dim_product.select("product_id", "product_key"),   on="product_id",  how="left")
    # Join to get region_key
    .join(dim_region,                                          on="region",      how="left")
    # Derive date_key
    .withColumn("date_key",  date_format(col("order_date"), "yyyyMMdd").cast("int"))
    .withColumn("ship_date_key", date_format(col("ship_date"), "yyyyMMdd").cast("int"))
    # Surrogate key
    .withColumn("sales_key", monotonically_increasing_id())
    .select(
        "sales_key",
        "date_key", "ship_date_key",
        "customer_key", "product_key", "region_key",
        "order_id", "order_line", "ship_mode",
        "quantity", "unit_price", "discount",
        "gross_sales", "discount_amt", "net_sales", "cogs", "profit",
        "_silver_loaded_at"
    )
)

print(f"fact_sales: {fact_sales.count():,} rows")
fact_sales.show(3)

In [ ]:
# ─────────────────────────────────────────────
# 7. Write Gold Tables to Fabric Warehouse via Spark connector
#    (Alternative: use stored procedures in warehouse/stored_procedures/)
# ─────────────────────────────────────────────

def write_to_warehouse(df, table_name, mode="overwrite"):
    """
    Write a Spark DataFrame to a Fabric Warehouse table.
    Fabric Warehouse supports direct Spark writes via the synapse connector.
    """
    df.write \
      .format("com.microsoft.sqlserver.jdbc.spark") \
      .option("url", WAREHOUSE_CONN) \
      .option("dbtable", f"dbo.{table_name}") \
      .option("reliabilityLevel", "BEST_EFFORT") \
      .option("tableLock", "true") \
      .mode(mode) \
      .save()
    print(f"✅ Written to warehouse: dbo.{table_name}")

write_to_warehouse(dim_customer, "dim_customer", mode="overwrite")
write_to_warehouse(dim_product,  "dim_product",  mode="overwrite")
write_to_warehouse(dim_date,     "dim_date",     mode="overwrite")
write_to_warehouse(dim_region,   "dim_region",   mode="overwrite")
write_to_warehouse(fact_sales,   "fact_sales",   mode="append")

print("\n🎉 Gold layer load complete — Star Schema populated!")